In [1]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path
import librosa
from torch.utils.data import Dataset, random_split, DataLoader
import gc


# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

from IPython.display import Audio, clear_output
import pandas as pd


In [2]:
os.listdir('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup')

['ESC-50-master',
 'sample_submission.csv',
 'genres_stems',
 'mashups',
 'test.csv']

In [3]:
test_df = pd.read_csv(INPUT_BASE+'/test.csv')

In [4]:
folder_index = {i:os.listdir('/'.join([INPUT_BASE, 'genres_stems', i])) for i in os.listdir(INPUT_BASE+'/genres_stems')}


STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

seed_everything(42)




In [5]:
genres = os.listdir(INPUT_BASE+'/genres_stems')

In [6]:
from tqdm import tqdm, trange

In [7]:
tf = INPUT_BASE + '/genres_stems/disco/'+folder_index['disco'][5]
os.listdir(tf)

['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

In [8]:
def find_tempo(waveform, sr):
    audio = waveform.squeeze().numpy().astype(np.float32)   # (T,)
    tempo, _ = librosa.beat.beat_track(y=audio, sr=sr)
    return max(float(np.atleast_1d(tempo)[0]), 1.0)

def match_tempo(stems, sr):
    target_length = stems[0].shape[1]
    tempos = [find_tempo(s, sr) for s in stems]
    target_tempo = float(np.median(tempos))                 # median > sorted[1]

    synced = []
    for i, (stem, src_tempo) in enumerate(zip(stems, tempos)):
        if abs(src_tempo - target_tempo) < 0.5:             # close enough, skip
            synced.append(stem)
            continue

        rate = src_tempo / target_tempo
        audio = stem.squeeze().numpy().astype(np.float32)   # (T,) — 1D for librosa

        stretched = librosa.effects.time_stretch(audio, rate=rate)

        if len(stretched) > target_length:
            stretched = stretched[:target_length]
        elif len(stretched) < target_length:
            stretched = np.pad(stretched, (0, target_length - len(stretched)))

        result = torch.from_numpy(stretched).unsqueeze(0)   # (1, T)
        del audio, stretched                                 # free immediately
        synced.append(result)

    return synced
        

In [9]:
target_sr = 16000
target_duration = 30
target_samples = target_duration * target_sr            



def load_aud(path, target_samples=target_samples, sr=target_sr):
    wav, orig_sr = torchaudio.load(path)
    if orig_sr != sr:
        wav = torchaudio.functional.resample(wav, orig_sr, sr)

    wav = wav.mean(dim=0)
    if wav.shape[0] > target_samples:
        start = torch.randint(0, wav.shape[0] - target_samples, (1,)).item()
        wav = wav[start:start + target_samples]
    elif wav.shape[0] < target_samples:
        wav = torch.nn.functional.pad(wav, (0, target_samples - wav.shape[0]))

    return wav 

In [10]:
def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=16000, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = os.listdir(INPUT_BASE+'/genres_stems')
    
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        clear_output()
        print(f'Genre: {genre}: {genres.index(genre)+1}/10')
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in trange(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            # stems = match_tempo(stems, target_sr)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup.mean(dim=0, keepdim=True)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                noise = noise.mean(dim=0, keepdim=True)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)
    
    gc.collect()

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=100)

Genre: pop: 10/10


100%|██████████| 100/100 [00:32<00:00,  3.08it/s]


In [11]:
Audio('/kaggle/working/synthetic_mashups/train/jazz/mashup_009.wav')

In [12]:
idx_g = dict(enumerate(set(genres)))
g_idx = {j:i for i,j in idx_g.items()}

In [13]:
import wandb

In [14]:
wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

class SyntheticDataset(Dataset):
    def __init__(self, output_dir):
        self.samples = []
        for genre in genres:
            genre_dir = Path(output_dir) / genre
            if not genre_dir.exists():
                continue
            for wav_path in sorted(genre_dir.glob("*.wav")):
                self.samples.append((str(wav_path), g_idx[genre]))
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        wav = load_aud(path)
        return wav, label

class TestMashupDataset(Dataset):
    def __init__(self, mashups_dir, test_csv):
        self.df = pd.read_csv(test_csv)
        self.mashups_dir = Path(mashups_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        fname = row.get("filename", f"{row['id']}.wav")
        wav   = load_aud(str(self.mashups_dir / fname))
        return wav, str(row["id"])

In [15]:
from transformers import ASTModel, ASTConfig, ASTFeatureExtractor
base_model = "MIT/ast-finetuned-audioset-10-10-0.4593"
fe = ASTFeatureExtractor.from_pretrained(base_model)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [34]:
import torch
from torch import nn
from transformers import ASTModel, ASTConfig, ASTFeatureExtractor
base_model = "MIT/ast-finetuned-audioset-10-10-0.4593"

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=10, backbone=base_model):
        super().__init__()
        self.backbone = ASTModel.from_pretrained(backbone)
        
        hidden = self.backbone.config.hidden_size 

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            # nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_values):
        # input_values shape: (batch, 1024, 128)
        outputs = self.backbone(input_values)
        
        # Use the [CLS] token (first token in the sequence) for classification
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        
        return self.classifier(pooled_output)


class SyntheticDataset(Dataset):
    def __init__(self, output_dir):
        self.samples = []
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(base_model)
        for genre in genres:
            genre_dir = Path(output_dir) / genre
            if not genre_dir.exists():
                continue
            for wav_path in sorted(genre_dir.glob("*.wav")):
                self.samples.append((str(wav_path), g_idx[genre]))
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        wav = load_aud(path)
        inputs = self.feature_extractor(wav, 
                                        sampling_rate=target_sr, 
                                        return_tensors='pt',
                                        max_length=1024)['input_values'].squeeze(0)
        return inputs, label

class TestMashupDataset(Dataset):
    def __init__(self, mashups_dir, test_csv):
        self.df = pd.read_csv(test_csv)
        self.mashups_dir = Path(mashups_dir)
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(base_model)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        fname = row.get("filename", f"{row['id']}.wav")
        wav   = load_aud(str(self.mashups_dir / fname))
        inputs = self.feature_extractor(wav, 
                                        sampling_rate=target_sr, 
                                        return_tensors='pt',
                                        max_length=1024)['input_values'].squeeze(0)
        return inputs, str(row["id"])

In [20]:
model = ASTClassifier()
model = torch.nn.DataParallel(model)
model = model.to('cuda')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

OUTPUT_PATH   = "/kaggle/working/synthetic_mashups/train"
SAVE_PATH     = "/kaggle/working/best_ast.pt"
BATCH_SIZE    = 8
EPOCHS        = 20
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.1
ACCUM_STEPS   = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"


def collate_fn(batch):
    inputs, labels = zip(*batch)

    inputs = torch.stack(inputs)               # (B, 1024, 128)
    labels = torch.tensor(labels, dtype=torch.long)
    return inputs, labels

In [35]:
wandb.init(
    project="messy-mashup",
    name="ast-finetuned",
    config={
        "backbone":     base_model,
        "batch_size":   BATCH_SIZE,
        "epochs":       EPOCHS,
        "lr":           LR,
        "accum_steps":  ACCUM_STEPS,
        "label_smoothing": 0.1,
    }
)

full_ds = SyntheticDataset(OUTPUT_PATH)
print(f"Total samples: {len(full_ds)}")

val_n   = max(1, int(len(full_ds) * VAL_SPLIT))
train_n = len(full_ds) - val_n
train_ds, val_ds = random_split(
    full_ds, [train_n, val_n],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True,
    collate_fn=collate_fn
)



Total samples: 1000


In [36]:
model     = ASTClassifier(num_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10           # 10% warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = GradScaler()

wandb.watch(model, log="gradients", log_freq=50)

best_f1     = 0.0
global_step = 0

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_55/2115142674.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
for epoch in range(1, EPOCHS + 1):

    # ── train ───────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (inputs, labels) in enumerate(
            tqdm(train_loader, desc=f"Ep {epoch:02d} train"), start=1):

        inputs = inputs.to(DEVICE)         # (B, 1024, 128)
        labels = labels.to(DEVICE)

        with autocast():
            logits = model(inputs)         # (B, 10)
            loss   = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            wandb.log({
                "train/batch_loss": loss.item() * ACCUM_STEPS,
                "train/lr":         scheduler.get_last_lr()[0],
            }, step=global_step)

    # ── validate ────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Ep {epoch:02d} val  "):
            inputs = inputs.to(DEVICE)
            with autocast():
                preds = model(inputs).argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())

    val_f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None,    zero_division=0)
    avg_loss     = running_loss / len(train_loader)

    epoch_metrics = {
        "epoch":              epoch,
        "train/epoch_loss":   avg_loss,
        "val/macro_f1":       val_f1,
    }
    for genre, f1 in zip(genres, per_class_f1):
        epoch_metrics[f"val/f1_{genre}"] = f1

    wandb.log(epoch_metrics, step=global_step)
    print(f"  loss: {avg_loss:.4f}  |  val macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  saved (best F1={best_f1:.4f})")

        artifact = wandb.Artifact("ast-best-model", type="model",
                                  description=f"val macro-F1: {best_f1:.4f}")
        artifact.add_file(SAVE_PATH)
        wandb.log_artifact(artifact)

    if epoch % 5 == 0:
        print(classification_report(
            all_labels, all_preds,
            target_names=genres, zero_division=0
        ))

    torch.cuda.empty_cache()

wandb.log({
    "val/confusion_matrix": wandb.plot.confusion_matrix(
        probs=None, y_true=all_labels,
        preds=all_preds, class_names=genres
    )
})

wandb.finish()
print(f"\nDone. Best val macro-F1: {best_f1:.4f}")


Ep 01 train:   0%|          | 0/113 [00:00<?, ?it/s]/tmp/ipykernel_55/3801687607.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Ep 01 val  :   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_55/3801687607.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Ep 01 val  : 100%|██████████| 13/13 [00:04<00:00,  2.71it/s]


  loss: 2.1289  |  val macro-F1: 0.6744
  saved (best F1=0.6744)


Ep 02 train:   0%|          | 0/113 [00:00<?, ?it/s]/tmp/ipykernel_55/3801687607.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Ep 02 val  :   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_55/3801687607.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Ep 02 val  : 100%|██████████| 13/13 [00:04<00:00,  2.96it/s]


  loss: 1.2907  |  val macro-F1: 0.7632
  saved (best F1=0.7632)


Ep 03 train:   0%|          | 0/113 [00:00<?, ?it/s]/tmp/ipykernel_55/3801687607.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Ep 03 train:   6%|▌         | 7/113 [00:05<01:20,  1.32it/s]

model(fe(load_aud('/kaggle/working/synthetic_mashups/train/jazz/mashup_009.wav'), sampling_rate=16000, return_tensors='pt')['input_values'].to('cuda'))

In [ ]:
FREEZE_LAYERS=8

In [78]:
from torch import nn
from transformers import HubertModel

basemodel = "facebook/hubert-base-ls960"

class HuBERTClassifier(nn.Module):
    def __init__(self, num_classes=10, backbone=basemodel, freeze_cnn=True):
        super().__init__()
        self.backbone = HubertModel.from_pretrained(backbone)

        if freeze_cnn:
            for p in self.backbone.feature_extractor.parameters():
                p.requires_grad = False

        # freeze bottom N of 12 transformer layers (unfreeze top 4 by default)
        for i, layer in enumerate(self.backbone.encoder.layers):
            if i<FREEZE_LAYERS:
                for p in layer.parameters():
                    p.requires_grad = False

        hidden = self.backbone.config.hidden_size  # 768 for hubert-base

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, 512),
            nn.GELU(),
            # nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, wav):
        # wav: (B, T) — directly from DataLoader, no squeeze needed
        out    = self.backbone(input_values=wav).last_hidden_state  # (B, frames, 768)
        mean_p = out.mean(dim=1)
        std_p  = out.std(dim=1)
        pooled = torch.cat([mean_p, std_p], dim=-1)                 # (B, 1536)
        return self.classifier(pooled)                           

In [29]:
full_ds = SyntheticDataset(OUTPUT_PATH)

In [30]:
train_ds, val_ds = random_split(
        full_ds, [450, 50],
        generator=torch.Generator().manual_seed(42)
    )

In [40]:
BATCH_SIZE=8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              pin_memory=True)

In [79]:
DEVICE='cuda'
model = HuBERTClassifier(num_classes=10).to(DEVICE)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [80]:
model = torch.nn.DataParallel(model)
model = torch.compile(model)

In [81]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report


In [82]:
LR = 1e-2

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)
scaler    = GradScaler('cuda')

In [83]:
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="23f3004489-dl-genai-project",
    # Set the wandb project where this run will be logged.
    project="dl-genai-project",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.01,
        "architecture": "hubert-base-ls960_freeze-8",
        "dataset": "main",
        "epochs": 20,
    },
)
wandb.watch(model, log="gradients", log_freq=50)

epoch,▁▂▃▃▄▅▆▆▇█
train/batch_loss,▃█▇▆█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_loss,█▃▁▁▁▁▁▁▁▁
train/lr,█████████████▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▃▃▃▃▂▂▁▁▁▁
val/f1_blues,▁▁▁▁▁▁▁▁▁▁
val/f1_classical,▁▁▁▁▁▁▁▁▁▁
val/f1_country,▁▁▁▁▁▁▁▁▁▁
val/f1_disco,▁▁▁▁▁▁▁▁▁▁
val/f1_hiphop,█▁▁▁▁▁▁▁▁▁
val/f1_jazz,▁▁▁▁▁▁▁▁▁▁
+5,...


In [84]:
best_f1     = 0.0
global_step = 0

In [85]:
SAVE_PATH = "best_model.pt"
EPOCHS = 10
ACCUM_STEPS = 8 # making gradient accumulate till model sees 64 samples

for epoch in range(1, EPOCHS + 1):

    # ── train ───────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (wavs, labels) in enumerate(
            tqdm(train_loader, desc=f"Ep {epoch:02d} train"), start=1):

        wavs, labels = wavs.to(DEVICE), labels.to(DEVICE)

        with autocast('cuda'):
            logits = model(wavs)
            loss   = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            global_step += 1

            wandb.log({
                "train/batch_loss": loss.item() * ACCUM_STEPS,
                "train/lr":         scheduler.get_last_lr()[0],
            }, step=global_step)

    scheduler.step()

    # ── validate ────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for wavs, labels in tqdm(val_loader, desc=f"Ep {epoch:02d} val  "):
            wavs  = wavs.to(DEVICE)
            with autocast('cuda'):
                preds = model(wavs).argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())

    val_f1      = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    avg_loss    = running_loss / len(train_loader)
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

    epoch_metrics = {
        "epoch":              epoch,
        "train/epoch_loss":   avg_loss,
        "val/macro_f1":       val_f1,
    }
    for genre, f1 in zip(genres, per_class_f1):
        epoch_metrics[f"val/f1_{genre}"] = f1

    wandb.log(epoch_metrics, step=global_step)
    print(f"  loss: {avg_loss:.4f}  |  val macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  saved  (best F1={best_f1:.4f})")
        artifact = wandb.Artifact("hubert-best-model", type="model",
                                  description=f"Best val macro-F1: {best_f1:.4f}")
        artifact.add_file(SAVE_PATH)
        wandb.log_artifact(artifact)

    if epoch % 5 == 0:
        print(classification_report(
            all_labels, all_preds,
            target_names=genres, zero_division=0
        ))

    torch.cuda.empty_cache()


Ep 01 val  : 100%|██████████| 7/7 [00:03<00:00,  1.87it/s]


  loss: 8.5781  |  val macro-F1: 0.0039
  saved  (best F1=0.0039)


Ep 02 val  : 100%|██████████| 7/7 [00:03<00:00,  1.85it/s]


  loss: 8.1849  |  val macro-F1: 0.0039


Ep 03 val  : 100%|██████████| 7/7 [00:04<00:00,  1.71it/s]


...

  loss: 2.3022  |  val macro-F1: 0.0039


Ep 09 val  : 100%|██████████| 7/7 [00:04<00:00,  1.70it/s]


  loss: 2.3024  |  val macro-F1: 0.0039


Ep 10 val  : 100%|██████████| 7/7 [00:03<00:00,  1.87it/s]


  loss: 2.4859  |  val macro-F1: 0.0039
              precision    recall  f1-score   support

       disco       0.00      0.00      0.00         3
       metal       0.00      0.00      0.00         6
      reggae       0.00      0.00      0.00         6
       blues       0.00      0.00      0.00         4
        rock       0.02      1.00      0.04         1
   classical       0.00      0.00      0.00         8
        jazz       0.00      0.00      0.00         6
      hiphop       0.00      0.00      0.00         7
     country       0.00      0.00      0.00         2
         pop       0.00      0.00      0.00         7

    accuracy                           0.02        50
   macro avg       0.00      0.10      0.00        50
weighted avg       0.00      0.02      0.00        50



In [76]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(all_labels, all_preds))

[[0 0 3 0 0 0 0 0 0 0]
 [0 0 6 0 0 0 0 0 0 0]
 [0 0 6 0 0 0 0 0 0 0]
 [0 0 4 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 8 0 0 0 0 0 0 0]
 [0 0 6 0 0 0 0 0 0 0]
 [0 0 7 0 0 0 0 0 0 0]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 7 0 0 0 0 0 0 0]]


In [51]:

print(classification_report(
    all_labels, all_preds,
    target_names=genres, zero_division=0
))

              precision    recall  f1-score   support

       disco       0.00      0.00      0.00         3
       metal       1.00      0.17      0.29         6
      reggae       0.12      1.00      0.22         6
       blues       0.00      0.00      0.00         4
        rock       0.00      0.00      0.00         1
   classical       0.00      0.00      0.00         8
        jazz       1.00      0.17      0.29         6
      hiphop       0.00      0.00      0.00         7
     country       0.00      0.00      0.00         2
         pop       0.00      0.00      0.00         7

    accuracy                           0.16        50
   macro avg       0.21      0.13      0.08        50
weighted avg       0.26      0.16      0.10        50



In [1]:
import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=16000):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '../working/synthetic_mashups/train'
OUTPUT_DIR = '../working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)




KeyboardInterrupt: 

In [1]:
import torch
import torch.nn as nn
import glob
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        return feature, label

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # TODO 1: Define the CNN Backbone using nn.Sequential
        # Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
        # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        self.cnn = None 
        
        # TODO 2: Define the RNN Component
        # Hint: Calculate the flattened feature size coming out of your CNN.
        # Original Mels = 128. After two MaxPool2d(2) layers, Mels = 128 / 4 = 32.
        # Channels = 64. So, Input Size = Channels * Mels = 2048.
        # Create a 1-layer Bidirectional LSTM with hidden_size=64 and batch_first=True.
        self.lstm = None
        
        # TODO 3: Define the Classifier
        # Create a Fully Connected (Linear) layer. 
        # Hint: Since the LSTM is bidirectional, the input features will be hidden_size * 2.
        self.fc = None

    def forward(self, x):
        """
        Input:
            x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
        Output:
            logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
        """
        
        # TODO 4: Pass 'x' through the CNN backbone
        # Expected shape after CNN: (Batch, Channels=64, Mels=32, Time)
        
        # TODO 5: Bridge the gap (Shape Manipulation)
        # Permute and reshape the CNN output to be compatible with the LSTM.
        # Extract b, c, f, t from the tensor shape.
        # Permute the dimensions to bring Time forward, then reshape to flatten Channels and Mels.
        # Target shape for LSTM: (Batch, Time_Steps, Channels * Mels)
        
        # TODO 6: Pass the reshaped sequence through the LSTM
        # The LSTM returns output and (hidden_state, cell_state). You only need the output.
        
        # TODO 7: Global Max Pooling over the time dimension
        # Collapse the sequence down to a single vector using torch.max() over the time dimension (dim=1).
        # Note: torch.max returns both values and indices. You only need the values.
        
        # TODO 8: Pass the pooled vector through the fully connected layer
        
        # return logits
        raise NotImplementedError("You need to implement the CRNN forward pass!")

In [45]:
#q1

sum([len(os.listdir('/kaggle/working/synthetic_mashups/train/'+i)) for i in os.listdir('/kaggle/working/synthetic_mashups/train/')])

500

In [81]:
#q2

aud = torchaudio.load('/kaggle/working/synthetic_mashups/train/disco/mashup_020.wav')
print(aud[0].shape)
del aud

torch.Size([2, 661500])


In [82]:
#Q3
aud = torch.load('../working/features/train/jazz/mashup_005.pt')
print(aud[0].shape)
del aud

torch.Size([128, 1292])


In [ ]:
#Q4
# Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
        # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
input_shp = [32, 1, 128, 1292]
# by adding pad 1 conv reduction for 3/3 kernel is avoided
maxpool_out_1 = [32, 1, 64, 646] #[input l&w/2]
maxpool_out_2 = [32, 1, 32, 323]

In [ ]:
#Q5

In [ ]:
#Q6
#spec has time in X and frequency in y which is pitch